# 🎵 Treinamento e Preparação de Dados Musical
### Projeto Integrador VI — Sistemas de Informação (PUC-Campinas, 2026) — Grupo B6

Notebook autocontido para o Google Colab (GPU T4). Faz o pipeline de dados (scripts 01 a 05 do repositório), treina o **Transformer** e a **LSTM**, gera estatísticas e **baixa os resultados** no final.

## Como usar
- **Primeira vez:** rode todas as células, na ordem.
- **A sessão caiu ou você reabriu o notebook:** rode todas as células de novo, na ordem. **Nada que já está pronto é refeito:**
  - dados processados: ficam no Drive e são pulados (a descompactação do `lmd_full.tar.gz` também);
  - pipeline interrompido no meio: retoma do último bloco de 2.000 arquivos gravado;
  - treino interrompido no meio: retoma da última época concluída;
  - treino já completo: não treina de novo nem repete a avaliação.
- **Já rodou a versão antiga deste notebook?** Os dados que ficaram no Drive (`treino/`, `validacao/`, `teste/`) são reaproveitados automaticamente, sem reler nenhum MIDI.
- **Quer refazer algo de propósito?** Use as chaves `REPROCESSAR_DADOS` e `NOVO_TREINO` da célula de configuração. O que existia é arquivado em `_arquivo/`, nunca apagado.

## O que fica no Google Drive (`Meus Drives/Colab Notebooks/SI-PI6-2026-G06/`)
| Pasta / arquivo | Conteúdo |
|---|---|
| `lmd_full.tar.gz` | dataset bruto (você coloca aqui; só é lido se os dados ainda não foram processados) |
| `dados_processados/blocos/` | resultado do pipeline por bloco de 2.000 MIDIs de entrada, com **todos os 100.000 arquivos** (aprovados com tokens; descartados com o motivo) |
| `dados_processados/consolidado/` | `treino`, `validacao` e `teste` em `.jsonl.gz` (é daqui que o treino lê) |
| `dados_processados/sequencias/vocabulario.json` | vocabulário |
| `dados_processados/janelas/` | janelas de treino já prontas (`.npy`), para o treino começar rápido |
| `dados_processados_lmd_100k.zip` | os dados no formato do repositório (`treino/`, `validacao/`, `teste/`, um `.json` por sequência) |
| `modelos/` | `modelo_*.pt` (melhor época) e `ultimo_*.pt` (para retomar) |
| `relatorios/` | histórico por época, resultados, gráfico, tabelas, `resumo_pipeline.json` e `manifesto_pipeline.csv` |
| `downloads/` | cópia do `.zip` com modelos e relatórios que é baixado no fim |

---
## 1. Verificação da GPU T4

In [ ]:
!nvidia-smi

---
## 2. Instalação das Dependências
Torch, NumPy, scikit-learn, pandas e matplotlib já vêm no Colab; instalá-los de novo só demora e pode trocar a versão do CUDA. Só falta o `pretty_midi`.

In [ ]:
!pip install -q pretty_midi

---
## 3. Montar o Google Drive
O `lmd_full.tar.gz` deve estar em `Meus Drives/Colab Notebooks/SI-PI6-2026-G06/` (só é necessário se os dados ainda não foram processados).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/SI-PI6-2026-G06')
TAR_DRIVE = DRIVE / 'lmd_full.tar.gz'
DRIVE.mkdir(parents=True, exist_ok=True)
print('Drive OK:', DRIVE)

---
## 4. Configuração (equivalente ao `configuracao/pipeline.yaml`)
Valores idênticos ao YAML e às constantes dos scripts 01 a 05. Se o YAML mudar, altere também esta célula.

**Chaves de controle** (no fim da célula):
- `REPROCESSAR_DADOS = True`: arquiva os dados processados e roda o pipeline do zero. Também é necessário se você mudar qualquer parâmetro do pipeline abaixo. **Volte para `False` assim que o pipeline terminar**, senão uma nova execução do notebook inteiro arquiva e refaz tudo outra vez.
- `NOVO_TREINO = True`: se o treino do modelo já está **completo**, arquiva o resultado e treina de novo. Treino **incompleto** é sempre retomado. Volte para `False` depois de iniciar, pelo mesmo motivo.

In [ ]:
import json, os, time, gzip, shutil, hashlib, subprocess, zipfile, math, csv
from datetime import datetime
import numpy as np

# piloto
PILOTO_ATIVO = True
PILOTO_QUANTIDADE = 100000
TAMANHO_BLOCO = 2000                 # arquivos de entrada por bloco gravado no Drive

# filtros (pipeline.yaml > filtros)
NOTAS_MINIMAS = 32
NOTAS_MAXIMAS = 2000
DURACAO_MINIMA_SEG = 15.0
DURACAO_MAXIMA_SEG = 600.0
PITCH_MINIMO = 36
PITCH_MAXIMO = 96

# normalização (pipeline.yaml > normalizacao)
GRADES_RITMICAS = [0.125, 0.25, 0.5, 1.0, 2.0]

# divisão (pipeline.yaml > divisao)
PROPORCAO_TREINO = 0.80
PROPORCAO_VALIDACAO = 0.10
SEMENTE = 42

# constantes dos scripts
TAMANHO_MINIMO_ARQUIVO_BYTES = 200   # script 02
LIMITE_SIMULTANEO_SEG = 0.05         # script 03 (50 ms)
LIMITE_PITCH_FORA_PCT = 0.05         # script 03
LIMITE_MELODIA_PLANA = 0.80          # script 03
LIMITE_REST_SEG = 0.01               # script 04
EXTENSOES = ('.mid', '.midi')
PESOS = {'notas': 0.3, 'registro': 0.2, 'variacao': 0.3, 'polifonia': 0.1, 'pausas': 0.1}  # script 03

# Tudo que, se mudar, invalida os dados já processados
CONFIG_DADOS = {
    'piloto_ativo': PILOTO_ATIVO, 'piloto_quantidade': PILOTO_QUANTIDADE,
    'notas_minimas': NOTAS_MINIMAS, 'notas_maximas': NOTAS_MAXIMAS,
    'duracao_minima_seg': DURACAO_MINIMA_SEG, 'duracao_maxima_seg': DURACAO_MAXIMA_SEG,
    'pitch_minimo': PITCH_MINIMO, 'pitch_maximo': PITCH_MAXIMO,
    'grades_ritmicas': GRADES_RITMICAS, 'proporcao_treino': PROPORCAO_TREINO,
    'proporcao_validacao': PROPORCAO_VALIDACAO, 'semente': SEMENTE,
    'tamanho_minimo_arquivo_bytes': TAMANHO_MINIMO_ARQUIVO_BYTES,
    'limite_simultaneo_seg': LIMITE_SIMULTANEO_SEG, 'limite_pitch_fora_pct': LIMITE_PITCH_FORA_PCT,
    'limite_melodia_plana': LIMITE_MELODIA_PLANA, 'limite_rest_seg': LIMITE_REST_SEG, 'pesos': PESOS,
}
CONFIG_HASH = hashlib.md5(json.dumps(CONFIG_DADOS, sort_keys=True).encode()).hexdigest()[:12]

# ---- chaves de controle ----
REPROCESSAR_DADOS = False   # True: arquiva os dados processados e refaz o pipeline do zero
NOVO_TREINO = False         # True: se o treino já está completo, arquiva e treina de novo
GERAR_ZIP_DADOS = True      # gera dados_processados_lmd_100k.zip no Drive (formato do repositório)
PERMITIR_CPU = False        # só para testes; treinar em CPU é inviável

try:
    import torch
    if not torch.cuda.is_available():
        print('AVISO: sem GPU. O pipeline de dados roda, mas o treino vai exigir GPU '
              '(Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU).')
except ImportError:
    pass

print(f'Piloto: {PILOTO_QUANTIDADE:,} arquivos (ativo={PILOTO_ATIVO}) | config dos dados: {CONFIG_HASH}')

---
## 5. Caminhos e utilitários
Toda gravação no Drive é feita em arquivo temporário e renomeada no fim, para uma queda de sessão nunca deixar um arquivo pela metade.

In [ ]:
PASTA_DADOS = DRIVE / 'dados_processados'
PASTA_RELATORIOS = DRIVE / 'relatorios'
PASTA_MODELOS = DRIVE / 'modelos'
pasta_seq = PASTA_DADOS / 'sequencias'
pasta_blocos = PASTA_DADOS / 'blocos'
pasta_cons = PASTA_DADOS / 'consolidado'
pasta_janelas = PASTA_DADOS / 'janelas'
pastas = {  # formato do pipeline antigo (um .json por sequência); só usado para importar dados antigos
    'treino': PASTA_DADOS / 'treino',
    'validacao': PASTA_DADOS / 'validacao',
    'teste': PASTA_DADOS / 'teste',
}
MARCADOR = PASTA_DADOS / 'pipeline_concluida.json'
SSD = Path('/content/cache_pi6')   # disco local do Colab (rápido); some quando a sessão cai
for p in [PASTA_DADOS, PASTA_RELATORIOS, PASTA_MODELOS, pasta_seq, SSD]:
    p.mkdir(parents=True, exist_ok=True)

def copiar_atomico(origem, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_name(destino.name + '.tmp')
    shutil.copyfile(origem, tmp)
    tmp.replace(destino)

def escrever_texto_atomico(destino, texto):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_name(destino.name + '.tmp')
    tmp.write_text(texto, encoding='utf-8')
    tmp.replace(destino)

def arquivar(itens, pasta_base):
    # Move para <pasta_base>/<data-hora>/ em vez de apagar
    itens = [Path(i) for i in itens if Path(i).exists()]
    if not itens:
        return None
    destino = Path(pasta_base) / datetime.now().strftime('%Y%m%d_%H%M%S')
    destino.mkdir(parents=True, exist_ok=True)
    for i in itens:
        shutil.move(str(i), str(destino / i.name))
    print(f'  arquivado em {destino}: {[i.name for i in itens]}')
    return destino

def ler_jsonl_gz(caminho):
    with gzip.open(caminho, 'rt', encoding='utf-8') as f:
        for linha in f:
            yield json.loads(linha)

def linha_json(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(',', ':')) + '\n'

print('Utilitários prontos.')

---
## 6. Estado dos dados
Decide o que precisa ser feito:
- **pronto**: os dados já foram processados (marcador `pipeline_concluida.json` no Drive). Nada é refeito.
- **importar_legado**: você já rodou o pipeline com a versão antiga deste notebook (pastas `treino/`, `validacao/`, `teste/` com um `.json` por sequência e `resumo_pipeline.json`). Os dados são **reaproveitados** (mesma divisão e mesmo vocabulário, então os modelos antigos continuam compatíveis) e convertidos para o formato novo, sem reler nenhum MIDI. Nesse caso o `manifesto_pipeline.csv` só terá os arquivos aprovados; o resumo com os descartes por motivo é preservado.
- **processar**: roda o pipeline (precisa do `lmd_full.tar.gz`), retomando do último bloco gravado.

In [ ]:
def contar_json(pasta):
    return sum(1 for a in pasta.glob('*.json') if a.stem != 'vocabulario') if pasta.exists() else 0

def detectar_legado():
    f_orig = PASTA_RELATORIOS / 'resumo_pipeline_original.json'   # existe se uma importação foi interrompida no fim
    f_resumo = f_orig if f_orig.exists() else PASTA_RELATORIOS / 'resumo_pipeline.json'
    if not f_resumo.exists() or not (pasta_seq / 'vocabulario.json').exists():
        print('  (sem dados antigos para reaproveitar: resumo_pipeline.json ou vocabulario.json ausente)')
        return None
    r = json.loads(f_resumo.read_text(encoding='utf-8'))
    if 'divisao' not in r or (f_resumo != f_orig and 'origem' in r):
        return None
    if PILOTO_ATIVO and r.get('piloto_quantidade') != PILOTO_QUANTIDADE:
        print(f"  (dados antigos não reaproveitados: foram gerados com piloto de {r.get('piloto_quantidade')} arquivos)")
        return None
    for nome, pasta in pastas.items():
        n = contar_json(pasta)
        if n != r['divisao'].get(nome, -1):
            print(f"  (dados antigos não reaproveitados: '{nome}' tem {n} arquivos no Drive, "
                  f"mas o resumo diz {r['divisao'].get(nome)})")
            return None
    return r

if REPROCESSAR_DADOS:
    print('REPROCESSAR_DADOS=True: arquivando dados processados anteriores...')
    arquivar([pasta_blocos, pasta_cons, pasta_janelas, MARCADOR, pasta_seq / 'vocabulario.json'],
             PASTA_DADOS / '_arquivo')
    print('  Depois que o pipeline terminar, volte REPROCESSAR_DADOS para False.')

ESTADO, RESUMO_LEGADO = 'processar', None
if MARCADOR.exists():
    marcador = json.loads(MARCADOR.read_text(encoding='utf-8'))
    if marcador.get('config_hash') != CONFIG_HASH:
        raise RuntimeError(
            f"Os dados prontos no Drive foram gerados com outra configuração ({marcador.get('config_hash')} "
            f"!= {CONFIG_HASH}). Se a mudança foi intencional, defina REPROCESSAR_DADOS = True; "
            "senão, restaure os valores da célula de configuração.")
    faltando = [n for n in pastas if not (pasta_cons / f'{n}.jsonl.gz').exists()]
    if faltando:
        raise RuntimeError(f'Marcador de dados prontos existe, mas faltam em {pasta_cons}: {faltando}. '
                           'Defina REPROCESSAR_DADOS = True.')
    ESTADO = 'pronto'
elif not REPROCESSAR_DADOS:
    RESUMO_LEGADO = detectar_legado()
    if RESUMO_LEGADO:
        ESTADO = 'importar_legado'

print('Estado dos dados:', ESTADO)
if ESTADO == 'pronto':
    print('  sequências:', marcador['divisao'], '| origem:', marcador.get('origem'), '| concluído em', marcador.get('concluido_em'))
elif ESTADO == 'importar_legado':
    print('  dados antigos completos encontrados; serão reaproveitados:', RESUMO_LEGADO['divisao'])
else:
    blocos_feitos = len(list(pasta_blocos.glob('bloco_*.jsonl.gz'))) if pasta_blocos.exists() else 0
    print(f'  pipeline será executado ({blocos_feitos} bloco(s) já gravado(s) no Drive serão reaproveitados)')

---
## 7. Descompactar o `lmd_full.tar.gz` no SSD do Colab
Só roda se o pipeline precisar rodar. Um arquivo `.extracao_completa` marca a extração concluída; uma extração interrompida é refeita.

In [ ]:
PASTA_BRUTOS = Path('/content/arquivos_brutos')
MARCA_EXTRACAO = PASTA_BRUTOS / '.extracao_completa'

if ESTADO != 'processar':
    print('Dados já processados: a descompactação do lmd_full.tar.gz não é necessária.')
else:
    assert TAR_DRIVE.exists(), f'Arquivo não encontrado no Drive: {TAR_DRIVE}'
    PASTA_BRUTOS.mkdir(parents=True, exist_ok=True)
    if MARCA_EXTRACAO.exists():
        print('Arquivos brutos já disponíveis no SSD do Colab.')
    else:
        print('Descompactando arquivos brutos (alguns minutos)...')
        t0 = time.time()
        subprocess.run(['tar', '-xzf', str(TAR_DRIVE), '-C', str(PASTA_BRUTOS)], check=True)
        MARCA_EXTRACAO.write_text(datetime.now().isoformat(), encoding='utf-8')
        print(f'Descompactação concluída em {time.time() - t0:.0f}s.')

---
## 8. Pipeline de Dados (scripts 01 a 05)
Etapas: **01** inventário (ordenação e piloto) → **02** validação (corrompido, sem notas, duração, tamanho) → **03** seleção de faixa por pontuação, monofonia e filtros finais (inclui melodia plana) → **04** tokenização → **05** divisão por `train_test_split` com semente fixa.

O piloto processa **100.000 arquivos MIDI** de entrada. Cada bloco de 2.000 arquivos é gravado no Drive assim que termina (`dados_processados/blocos/`), com uma linha por arquivo de entrada: aprovado (com os tokens) ou descartado (com o motivo). Se a sessão cair, basta rodar de novo: os blocos já gravados são pulados.

In [ ]:
import pretty_midi
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

DESCARTES_CHAVES = ['arquivo_minimo', 'corrompido', 'sem_notas', 'curto_demais', 'longo_demais',
                    'sem_faixa', 'notas_insuficientes', 'notas_excessivas', 'melodia_plana']

def quantizar(duracao_beats):
    grades = np.array(GRADES_RITMICAS)
    return f'{grades[int(np.argmin(np.abs(grades - duracao_beats)))]:.2f}'

def bpm_do_midi(pm):
    tempos = pm.get_tempo_changes()
    return float(tempos[1][0]) if len(tempos[1]) > 0 else 120.0

def pontuacao_faixa(inst, duracao_total_midi):
    notas = inst.notes
    num = len(notas)
    pitches = [n.pitch for n in notas]
    pitch_medio = np.mean(pitches)
    variacao = np.std(pitches)
    ordenadas = sorted(notas, key=lambda n: n.start)
    sobrepostos = sum(1 for i in range(1, num) if ordenadas[i].start < ordenadas[i - 1].end - 0.001)
    pct_polifonia = sobrepostos / num
    duracao_total = ordenadas[-1].end - ordenadas[0].start
    tempo_com_nota = sum(n.end - n.start for n in ordenadas)
    pct_pausas = max(0.0, 1.0 - tempo_com_nota / duracao_total) if duracao_total > 0 else 1.0
    norm_notas = min(num / 500.0, 1.0)
    norm_registro = min(max((pitch_medio - 36) / 60.0, 0.0), 1.0)
    norm_var = min(variacao / 12.0, 1.0)
    return (PESOS['notas'] * norm_notas + PESOS['registro'] * norm_registro
            + PESOS['variacao'] * norm_var - PESOS['polifonia'] * pct_polifonia
            - PESOS['pausas'] * pct_pausas)

def selecionar_faixa(pm):
    melhor, melhor_score = None, -999.0
    duracao = pm.get_end_time()
    for inst in pm.instruments:
        if inst.is_drum or len(inst.notes) < NOTAS_MINIMAS:
            continue
        if duracao < DURACAO_MINIMA_SEG:
            continue
        score = pontuacao_faixa(inst, duracao)
        if score > melhor_score:
            melhor, melhor_score = inst, score
    return melhor

def monofonia(notas):
    ordenadas = sorted(notas, key=lambda n: n.start)
    resultado, grupo = [], [ordenadas[0]]
    for nota in ordenadas[1:]:
        if nota.start - grupo[-1].start < LIMITE_SIMULTANEO_SEG:
            grupo.append(nota)
        else:
            resultado.append(max(grupo, key=lambda n: n.pitch))
            grupo = [nota]
    resultado.append(max(grupo, key=lambda n: n.pitch))
    return resultado

def filtros_finais(notas):
    fora = [n for n in notas if not (PITCH_MINIMO <= n.pitch <= PITCH_MAXIMO)]
    if len(fora) / len(notas) > LIMITE_PITCH_FORA_PCT:
        notas = [n for n in notas if PITCH_MINIMO <= n.pitch <= PITCH_MAXIMO]
    if len(notas) < NOTAS_MINIMAS:
        return None, 'NOTAS_INSUFICIENTES'
    if len(notas) > NOTAS_MAXIMAS:
        return None, 'NOTAS_EXCESSIVAS'
    contagem = {}
    for n in notas:
        contagem[n.pitch] = contagem.get(n.pitch, 0) + 1
    if sum(sorted(contagem.values(), reverse=True)[:2]) / len(notas) > LIMITE_MELODIA_PLANA:
        return None, 'MELODIA_PLANA'
    return notas, None

def tokenizar(notas, bpm):
    spb = bpm / 60.0
    tokens = []
    for i, n in enumerate(notas):
        if i > 0:
            gap = n.start - notas[i - 1].end
            if gap > LIMITE_REST_SEG:
                tokens.append(f'REST_{quantizar(gap * spb)}')
        tokens += [f'NOTE_{n.pitch}', f'DURATION_{quantizar((n.end - n.start) * spb)}']
    return tokens

def processar_arquivo(caminho):
    # Retorna (status, dados). status = 'aprovado' ou a chave de descarte.
    if caminho.stat().st_size < TAMANHO_MINIMO_ARQUIVO_BYTES:
        return 'arquivo_minimo', None
    try:
        pm = pretty_midi.PrettyMIDI(str(caminho))
    except Exception:
        return 'corrompido', None
    duracao = pm.get_end_time()
    if sum(len(i.notes) for i in pm.instruments) == 0:
        return 'sem_notas', None
    if duracao < DURACAO_MINIMA_SEG:
        return 'curto_demais', None
    if duracao > DURACAO_MAXIMA_SEG:
        return 'longo_demais', None
    faixa = selecionar_faixa(pm)
    if faixa is None:
        return 'sem_faixa', None
    notas, motivo = filtros_finais(monofonia(faixa.notes))
    if motivo:
        return motivo.lower(), None
    bpm = bpm_do_midi(pm)
    tokens = tokenizar(notas, bpm)
    if not tokens:
        return 'sem_faixa', None
    return 'aprovado', (round(bpm, 2), round(duracao, 3), tokens)

def nome_bloco(i):
    return f'bloco_{i:04d}.jsonl.gz'

def processar_pipeline_em_blocos():
    # Etapas 01 a 04, gravando cada bloco no Drive. Retorna o número de blocos.
    todos = sorted(p for p in PASTA_BRUTOS.rglob('*') if p.suffix.lower() in EXTENSOES)
    print(f'MIDIs brutos localizados: {len(todos):,}')
    if PILOTO_ATIVO:
        todos = todos[:PILOTO_QUANTIDADE]
    print(f'Arquivos de entrada do piloto: {len(todos):,}')
    if not todos:
        raise RuntimeError(f'Nenhum MIDI encontrado em {PASTA_BRUTOS}.')

    cfg = {'config_hash': CONFIG_HASH, 'tamanho_bloco': TAMANHO_BLOCO, 'arquivos_entrada': len(todos),
           'primeiro_arquivo': todos[0].name, 'ultimo_arquivo': todos[-1].name}
    pasta_blocos.mkdir(parents=True, exist_ok=True)
    f_cfg = pasta_blocos / 'config_blocos.json'
    if f_cfg.exists():
        salvo = json.loads(f_cfg.read_text(encoding='utf-8'))
        if salvo != cfg:
            raise RuntimeError(
                'Os blocos já gravados no Drive foram gerados com outra configuração ou outra lista de arquivos '
                f'de entrada.\n  gravado: {salvo}\n  atual:   {cfg}\n'
                'Restaure a configuração original ou defina REPROCESSAR_DADOS = True.')
    else:
        escrever_texto_atomico(f_cfg, json.dumps(cfg, ensure_ascii=False, indent=2))

    n_blocos = math.ceil(len(todos) / TAMANHO_BLOCO)
    feitos = [i for i in range(n_blocos) if (pasta_blocos / nome_bloco(i)).exists()]
    if feitos:
        print(f'{len(feitos)} de {n_blocos} blocos já estão no Drive; retomando.')
    pasta_ssd = SSD / 'blocos'
    pasta_ssd.mkdir(parents=True, exist_ok=True)

    ja_feitos = sum(len(todos[i * TAMANHO_BLOCO:(i + 1) * TAMANHO_BLOCO]) for i in feitos)
    barra = tqdm(total=len(todos), initial=ja_feitos, desc='Processando MIDIs')
    for i in range(n_blocos):
        if i in feitos:
            continue
        fatia = todos[i * TAMANHO_BLOCO:(i + 1) * TAMANHO_BLOCO]
        saida = pasta_ssd / nome_bloco(i)
        with gzip.open(saida, 'wt', encoding='utf-8', compresslevel=3) as f:
            for caminho in fatia:
                status, dados = processar_arquivo(caminho)
                reg = {'id': caminho.stem, 'status': status}
                if dados:
                    reg.update({'bpm': dados[0], 'duracao_seg': dados[1], 'tokens': dados[2]})
                f.write(linha_json(reg))
                barra.update(1)
        copiar_atomico(saida, pasta_blocos / nome_bloco(i))
        barra.set_postfix_str(f'bloco {i + 1}/{n_blocos} gravado no Drive')
    barra.close()
    return n_blocos

if ESTADO == 'processar':
    N_BLOCOS = processar_pipeline_em_blocos()
    print(f'Pipeline concluído: {N_BLOCOS} blocos no Drive ({pasta_blocos}).')
else:
    print('Pipeline de MIDIs não precisa rodar.')

---
## 9. Consolidação: vocabulário, divisão treino/validação/teste e relatório
Etapa 04 (vocabulário em ordem de aparição, especiais no fim) e etapa 05 (duas chamadas de `train_test_split` com a mesma semente). O resultado vai para `dados_processados/consolidado/` e o marcador `pipeline_concluida.json` é gravado por último.

In [ ]:
def gravar_vocabulario(vocabulario):
    for especial in ['<PAD>', '<START>', '<END>']:
        if especial not in vocabulario:
            vocabulario[especial] = len(vocabulario)
    vj = {
        'token_para_id': vocabulario,
        'id_para_token': {str(v): k for k, v in vocabulario.items()},
        'tamanho_vocabulario': len(vocabulario),
        'tokens_especiais': {'PAD': '<PAD>', 'START': '<START>', 'END': '<END>'},
    }
    escrever_texto_atomico(pasta_seq / 'vocabulario.json', json.dumps(vj, ensure_ascii=False, indent=2))
    return vj

def gravar_manifesto(linhas):
    # Uma linha por arquivo de entrada do pipeline
    tmp = SSD / 'manifesto_pipeline.csv'
    with open(tmp, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['arquivo_md5', 'status', 'bpm', 'duracao_seg', 'num_tokens'])
        w.writerows(linhas)
    copiar_atomico(tmp, PASTA_RELATORIOS / 'manifesto_pipeline.csv')

def publicar_consolidado():
    for nome in pastas:
        copiar_atomico(SSD / 'consolidado' / f'{nome}.jsonl.gz', pasta_cons / f'{nome}.jsonl.gz')

def finalizar_pipeline(origem, resumo_base, divisao, vocab_tam):
    resumo = {**resumo_base, 'origem': origem, 'divisao': divisao, 'vocabulario_tamanho': vocab_tam,
              'sequencias_aprovadas': sum(divisao.values())}
    escrever_texto_atomico(PASTA_RELATORIOS / 'resumo_pipeline.json', json.dumps(resumo, ensure_ascii=False, indent=2))
    escrever_texto_atomico(MARCADOR, json.dumps({
        'config_hash': CONFIG_HASH, 'origem': origem, 'divisao': divisao,
        'concluido_em': datetime.now().isoformat(timespec='seconds')}, ensure_ascii=False, indent=2))
    return resumo

def consolidar_blocos(n_blocos):
    vocabulario, md5s, manifesto = {}, [], []
    descartes = {k: 0 for k in DESCARTES_CHAVES}
    total_entrada = 0
    for i in tqdm(range(n_blocos), desc='Lendo blocos (vocabulário e divisão)'):
        for r in ler_jsonl_gz(pasta_blocos / nome_bloco(i)):
            total_entrada += 1
            if r['status'] != 'aprovado':
                descartes[r['status']] += 1
                manifesto.append((r['id'], r['status'], '', '', ''))
                continue
            md5s.append(r['id'])
            for tok in r['tokens']:
                if tok not in vocabulario:
                    vocabulario[tok] = len(vocabulario)
            manifesto.append((r['id'], 'aprovado', r['bpm'], r['duracao_seg'], len(r['tokens'])))
    vj = gravar_vocabulario(vocabulario)

    md5_treino, md5_resto = train_test_split(md5s, test_size=1.0 - PROPORCAO_TREINO, random_state=SEMENTE, shuffle=True)
    pct_val_do_resto = PROPORCAO_VALIDACAO / (1.0 - PROPORCAO_TREINO)
    md5_val, md5_teste = train_test_split(md5_resto, test_size=1.0 - pct_val_do_resto, random_state=SEMENTE, shuffle=True)
    split_de = {}
    for nome, lista in {'treino': md5_treino, 'validacao': md5_val, 'teste': md5_teste}.items():
        for m in lista:
            split_de[m] = nome

    (SSD / 'consolidado').mkdir(parents=True, exist_ok=True)
    destinos = {n: gzip.open(SSD / 'consolidado' / f'{n}.jsonl.gz', 'wt', encoding='utf-8', compresslevel=3) for n in pastas}
    try:
        for i in tqdm(range(n_blocos), desc='Gravando treino/validação/teste'):
            for r in ler_jsonl_gz(pasta_blocos / nome_bloco(i)):
                if r['status'] == 'aprovado':
                    destinos[split_de[r['id']]].write(linha_json({
                        'md5': r['id'], 'bpm': r['bpm'], 'duracao_seg': r['duracao_seg'],
                        'num_tokens': len(r['tokens']), 'tokens': r['tokens']}))
    finally:
        for f in destinos.values():
            f.close()
    publicar_consolidado()
    gravar_manifesto(manifesto)
    divisao = {'treino': len(md5_treino), 'validacao': len(md5_val), 'teste': len(md5_teste)}
    return finalizar_pipeline('pipeline', {
        'piloto_quantidade': PILOTO_QUANTIDADE, 'arquivos_entrada': total_entrada, 'descartes': descartes,
        'tamanho_bloco': TAMANHO_BLOCO, 'blocos': n_blocos}, divisao, len(vj['token_para_id']))

def consolidar_legado(resumo_legado):
    # Reaproveita os .json (um por sequência) do pipeline antigo sem reler nenhum MIDI.
    (SSD / 'consolidado').mkdir(parents=True, exist_ok=True)
    manifesto = []
    for nome, pasta in pastas.items():
        arqs = sorted(a for a in pasta.glob('*.json') if a.stem != 'vocabulario')
        saida = SSD / 'consolidado' / f'{nome}.jsonl.gz'
        with ThreadPoolExecutor(max_workers=16) as ex, gzip.open(saida, 'wt', encoding='utf-8', compresslevel=3) as f:
            for texto in tqdm(ex.map(lambda a: a.read_text(encoding='utf-8'), arqs), total=len(arqs),
                              desc=f'Importando {nome}'):
                d = json.loads(texto)
                f.write(linha_json({'md5': d['md5'], 'bpm': d.get('bpm'), 'duracao_seg': d.get('duracao_seg'),
                                    'num_tokens': len(d['tokens']), 'tokens': d['tokens']}))
                manifesto.append((d['md5'], 'aprovado', d.get('bpm', ''), d.get('duracao_seg', ''), len(d['tokens'])))
    publicar_consolidado()
    gravar_manifesto(manifesto)
    vj = json.loads((pasta_seq / 'vocabulario.json').read_text(encoding='utf-8'))
    escrever_texto_atomico(PASTA_RELATORIOS / 'resumo_pipeline_original.json', json.dumps(resumo_legado, ensure_ascii=False, indent=2))
    base = {k: v for k, v in resumo_legado.items() if k not in ('divisao', 'vocabulario_tamanho', 'sequencias_aprovadas')}
    return finalizar_pipeline('importado_legado', base, resumo_legado['divisao'], vj['tamanho_vocabulario'])

if ESTADO == 'processar':
    resumo = consolidar_blocos(N_BLOCOS)
elif ESTADO == 'importar_legado':
    resumo = consolidar_legado(RESUMO_LEGADO)
else:
    resumo = json.loads((PASTA_RELATORIOS / 'resumo_pipeline.json').read_text(encoding='utf-8'))
ESTADO = 'pronto'

print()
print(f"Sequências aprovadas: {resumo['sequencias_aprovadas']:,} de {resumo.get('arquivos_entrada', '?')} arquivos de entrada")
print('Descartes:', resumo.get('descartes'))
print('Treino / validação / teste:', resumo['divisao'])
print('Vocabulário:', resumo['vocabulario_tamanho'], 'tokens')

---
## 10. Dados no formato do repositório (opcional)
Gera `dados_processados_lmd_100k.zip` no Drive, com um `.json` por sequência em `treino/`, `validacao/` e `teste/` (o formato que `scripts/treinar_neural.py` espera). Para usar no PC: extraia na raiz do projeto. Só é gerado se ainda não existir.

In [ ]:
ZIP_DADOS = DRIVE / 'dados_processados_lmd_100k.zip'
if not GERAR_ZIP_DADOS:
    print('GERAR_ZIP_DADOS=False: pulado.')
elif ZIP_DADOS.exists():
    print('Já existe:', ZIP_DADOS)
else:
    tmp_zip = SSD / ZIP_DADOS.name
    with zipfile.ZipFile(tmp_zip, 'w', zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
        zf.write(pasta_seq / 'vocabulario.json', 'dados_processados/sequencias/vocabulario.json')
        for nome in pastas:
            for r in tqdm(ler_jsonl_gz(pasta_cons / f'{nome}.jsonl.gz'), desc=f'zip {nome}',
                          total=resumo['divisao'][nome]):
                zf.writestr(f"dados_processados/{nome}/{r['md5']}.json", json.dumps(r, ensure_ascii=False, separators=(',', ':')))
    copiar_atomico(tmp_zip, ZIP_DADOS)
    print(f'Gerado: {ZIP_DADOS} ({ZIP_DADOS.stat().st_size / 1e6:.0f} MB)')

---
## 11. Funções de Treino, Avaliação e Geração (Transformer e LSTM)
Define o carregamento das janelas, as arquiteturas (iguais às de `src/modelos/`), as métricas e as rotinas de geração.

As janelas de treino (`.npy`) são calculadas uma vez e guardadas no Drive; nas próximas sessões o treino começa em segundos.

Métricas calculadas em cada conjunto (treino, validação e teste), sobre tokens que não são `<PAD>`:
- **Loss** (cross-entropy) e **perplexidade** (exp(loss))
- **Acurácia** (token previsto = token real) e **Top-5**
- **Precisão, Recall e F1 macro** (média sobre as classes presentes no alvo)
- **AUC-ROC** one-vs-rest macro (amostra de até 200 mil tokens, só no teste)

Geração: 50 melodias por modelo, com as mesmas regras de amostragem de `src/modelos/transformer_model.py` (gramática NOTE→DURATION→NOTE/REST, penalidade de repetição, top-k/top-p). A LSTM não tem gramática no repositório, então é gerada com top-k simples, como em `src/modelos/lstm_model.py`. As estatísticas de melodia comparam as gerações com as melodias reais do teste.

**Retomada:** a cada época o estado completo (modelo, otimizador, scheduler e histórico) é gravado em `modelos/ultimo_<modelo>.pt`. Se a sessão cair, rode o notebook de novo: o treino continua da última época concluída. Um modelo de uma execução antiga, já completo (tem `resultados_<modelo>.json` com todas as épocas), é reaproveitado e não é treinado de novo.

In [ ]:
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
if device.type == 'cpu' and not PERMITIR_CPU:
    raise RuntimeError('Sem GPU. Use Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU '
                       'e rode as células de novo (os dados já processados no Drive são reaproveitados).')

vocab_json = json.loads((pasta_seq / 'vocabulario.json').read_text(encoding='utf-8'))
token_para_id = vocab_json['token_para_id']
id_para_token = vocab_json['id_para_token']
VOCAB_SIZE = len(token_para_id)
VOCAB_HASH = hashlib.md5(json.dumps(token_para_id, sort_keys=True).encode()).hexdigest()[:8]
PAD_ID = token_para_id['<PAD>']
START_ID = token_para_id['<START>']
END_ID = token_para_id['<END>']
ID_DUR_MIN = token_para_id.get('DURATION_0.12')

# Hiperparâmetros (iguais a scripts/treinar_neural.py)
SEQ_LEN = 128
BATCH_SIZE = 32
LR = 1e-3
D_MODEL, NHEAD, NUM_LAYERS, DIM_FF, MAX_SEQ_LEN = 256, 4, 4, 512, 1024
DROPOUT_TRANSFORMER, DROPOUT_LSTM = 0.1, 0.2
EPOCAS_TRANSFORMER, EPOCAS_LSTM = 15, 20
N_MELODIAS_GERADAS = 50
TOKENS_POR_MELODIA = 256

def carregar_janelas(split):
    nome = f'janelas_{split}_L{SEQ_LEN}_{VOCAB_HASH}.npy'
    ssd, drv = SSD / nome, pasta_janelas / nome
    if ssd.exists():
        return np.load(ssd)
    if drv.exists():
        shutil.copyfile(drv, ssd)
        return np.load(ssd)
    janelas = []
    total = json.loads((PASTA_RELATORIOS / 'resumo_pipeline.json').read_text(encoding='utf-8'))['divisao'][split]
    for r in tqdm(ler_jsonl_gz(pasta_cons / f'{split}.jsonl.gz'), total=total, desc=f'Janelas {split}'):
        tokens = r['tokens']
        if len(tokens) < 8:
            continue
        ids = [START_ID] + [token_para_id.get(t, PAD_ID) for t in tokens] + [END_ID]
        for i in range(0, max(1, len(ids) - SEQ_LEN), max(1, SEQ_LEN // 2)):
            janela = ids[i : i + SEQ_LEN + 1]
            if len(janela) > 4:
                janelas.append(np.array(janela + [PAD_ID] * (SEQ_LEN + 1 - len(janela)), dtype=np.int16))
    arr = np.stack(janelas) if janelas else np.zeros((0, SEQ_LEN + 1), dtype=np.int16)
    np.save(ssd, arr)
    copiar_atomico(ssd, drv)
    return arr

print('Carregando janelas (do Drive, se já calculadas)...')
dados_treino = carregar_janelas('treino')
dados_val = carregar_janelas('validacao')
dados_teste = carregar_janelas('teste')
print(f'Janelas | treino: {len(dados_treino):,} | validação: {len(dados_val):,} | teste: {len(dados_teste):,}')
if len(dados_treino) == 0:
    raise ValueError('Sem dados de treino. Verifique as células 6 a 9.')

class JanelasDataset(Dataset):
    def __init__(self, dados):
        self.dados = dados

    def __len__(self):
        return len(self.dados)

    def __getitem__(self, i):
        w = self.dados[i].astype(np.int64)
        return torch.from_numpy(w[:-1]), torch.from_numpy(w[1:])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout, max_seq_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                           dropout=dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=x.device), diagonal=1)
        embeds = self.embedding(x) * math.sqrt(self.d_model)
        out = self.transformer(self.pos_encoder(embeds), mask=mask, is_causal=True)
        return self.fc_out(out)

class MelodyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embedding(x), hidden)
        return self.fc(self.dropout(out)), hidden

def criar_modelo(nome):
    if nome == 'transformer':
        return MusicTransformer(VOCAB_SIZE, D_MODEL, NHEAD, NUM_LAYERS, DIM_FF, DROPOUT_TRANSFORMER, MAX_SEQ_LEN).to(device)
    return MelodyLSTM(VOCAB_SIZE, D_MODEL, DIM_FF, NUM_LAYERS, DROPOUT_LSTM).to(device)

def logits_de(modelo, x):
    saida = modelo(x)
    return saida[0] if isinstance(saida, tuple) else saida

def auc_ovr(probs, y):
    aucs = []
    for c in np.unique(y):
        positivos = y == c
        if positivos.sum() == 0 or positivos.sum() == len(y):
            continue
        aucs.append(roc_auc_score(positivos, probs[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')

def avaliar(modelo, dl, calcular_auc=False, max_auc=200000):
    modelo.eval()
    criterio_soma = nn.CrossEntropyLoss(ignore_index=PAD_ID, reduction='sum')
    perda, total, acertos, top5 = 0.0, 0, 0, 0
    tp = torch.zeros(VOCAB_SIZE)
    pred_c = torch.zeros(VOCAB_SIZE)
    true_c = torch.zeros(VOCAB_SIZE)
    amostra_p, amostra_y, n_auc = [], [], 0
    with torch.no_grad():
        for x, y in dl:
            logits = logits_de(modelo, x.to(device)).reshape(-1, VOCAB_SIZE)
            y = y.to(device).reshape(-1)
            m = y != PAD_ID
            logits, y = logits[m], y[m]
            if y.numel() == 0:
                continue
            perda += criterio_soma(logits, y).item()
            total += y.numel()
            pred = logits.argmax(-1)
            certo = pred == y
            acertos += certo.sum().item()
            top5 += (logits.topk(5, dim=-1).indices == y[:, None]).any(-1).sum().item()
            tp += torch.bincount(y[certo], minlength=VOCAB_SIZE).cpu().float()
            pred_c += torch.bincount(pred, minlength=VOCAB_SIZE).cpu().float()
            true_c += torch.bincount(y, minlength=VOCAB_SIZE).cpu().float()
            if calcular_auc and n_auc < max_auc:
                amostra_p.append(F.softmax(logits, dim=-1).cpu().numpy())
                amostra_y.append(y.cpu().numpy())
                n_auc += y.numel()
    if total == 0:
        raise ValueError('Conjunto de avaliação vazio.')
    presentes = true_c > 0
    precisao_c = tp / pred_c.clamp(min=1)
    recall_c = tp / true_c.clamp(min=1)
    f1_c = 2 * precisao_c * recall_c / (precisao_c + recall_c).clamp(min=1e-12)
    resultado = {
        'loss': perda / total,
        'perplexidade': math.exp(perda / total),
        'acuracia': acertos / total,
        'top5': top5 / total,
        'precisao_macro': precisao_c[presentes].mean().item(),
        'recall_macro': recall_c[presentes].mean().item(),
        'f1_macro': f1_c[presentes].mean().item(),
        'tokens_avaliados': total,
    }
    if calcular_auc:
        resultado['auc_roc_ovr'] = auc_ovr(np.concatenate(amostra_p)[:max_auc], np.concatenate(amostra_y)[:max_auc])
    return resultado

def salvar_checkpoint(nome, modelo, caminho):
    ckpt = {'state_dict': modelo.state_dict(), 'vocab_size': VOCAB_SIZE, 'vocabulario': vocab_json}
    if nome == 'transformer':
        ckpt.update({'d_model': D_MODEL, 'nhead': NHEAD, 'num_layers': NUM_LAYERS,
                     'dim_feedforward': DIM_FF, 'max_seq_len': MAX_SEQ_LEN})
    else:
        ckpt.update({'embed_dim': D_MODEL, 'hidden_dim': DIM_FF, 'num_layers': NUM_LAYERS})
    tmp = SSD / f'tmp_{nome}.pt'
    torch.save(ckpt, tmp)
    copiar_atomico(tmp, caminho)

def salvar_historico_csv(nome, historico):
    tmp = SSD / f'historico_{nome}.csv'
    with open(tmp, 'w', newline='', encoding='utf-8') as f:
        escritor = csv.DictWriter(f, fieldnames=list(historico[0].keys()))
        escritor.writeheader()
        escritor.writerows(historico)
    copiar_atomico(tmp, PASTA_RELATORIOS / f'historico_{nome}.csv')

def treinar_modelo(nome, epocas):
    # Retorna (historico, treinou_agora)
    caminho_melhor = PASTA_MODELOS / f'modelo_{nome}.pt'
    caminho_ultimo = PASTA_MODELOS / f'ultimo_{nome}.pt'
    f_resultados = PASTA_RELATORIOS / f'resultados_{nome}.json'
    arquivos_do_modelo = [caminho_ultimo, caminho_melhor, PASTA_RELATORIOS / f'historico_{nome}.csv', f_resultados]
    estado = None
    if caminho_ultimo.exists():
        estado = torch.load(caminho_ultimo, map_location=device, weights_only=False)
        if estado['vocab_hash'] != VOCAB_HASH:
            raise RuntimeError(f'O treino salvo de {nome} usa outro vocabulário (os dados foram reprocessados?). '
                               'Defina NOVO_TREINO = True para arquivar o modelo antigo e treinar de novo.')
        completo = estado['epoca'] >= estado['epocas_total']
        if completo and NOVO_TREINO:
            print(f'NOVO_TREINO=True e o treino de {nome} está completo: arquivando e recomeçando.')
            arquivar(arquivos_do_modelo, PASTA_MODELOS / '_arquivo')
            estado = None
        elif estado['epocas_total'] != epocas:
            raise RuntimeError(f"O treino salvo de {nome} foi iniciado com {estado['epocas_total']} épocas, mas a "
                               f'configuração atual tem {epocas}. Restaure o valor original para retomar.')
    elif caminho_melhor.exists():
        r = json.loads(f_resultados.read_text(encoding='utf-8')) if f_resultados.exists() else None
        if r and len(r['historico']) == epocas and not NOVO_TREINO:
            ck = torch.load(caminho_melhor, map_location='cpu', weights_only=False)
            if ck.get('vocabulario', {}).get('token_para_id') == token_para_id:
                print(f'{nome}: treino completo de uma execução anterior ({epocas} épocas); reaproveitado, sem retreinar.')
                return r['historico'], False
            print(f'{nome}: o modelo anterior foi treinado com outro vocabulário que o dos dados atuais.')
        print(f'{nome}: há um modelo anterior sem estado de retomada. Arquivando antes de treinar.')
        arquivar(arquivos_do_modelo, PASTA_MODELOS / '_arquivo')

    torch.manual_seed(SEMENTE)
    modelo = criar_modelo(nome)
    dl_treino = DataLoader(JanelasDataset(dados_treino), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    dl_val = DataLoader(JanelasDataset(dados_val), batch_size=BATCH_SIZE, shuffle=False)
    criterio = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    otim = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(otim, T_max=epocas)

    historico, melhor, ep_inicio = [], float('inf'), 1
    if estado is not None:
        modelo.load_state_dict(estado['modelo'])
        otim.load_state_dict(estado['otimizador'])
        sched.load_state_dict(estado['scheduler'])
        historico, melhor, ep_inicio = estado['historico'], estado['melhor_val'], estado['epoca'] + 1
        if ep_inicio > epocas:
            print(f'Treino de {nome} já está completo ({epocas} épocas); nada a treinar.')
            return historico, False
        print(f'Retomando {nome.upper()} da época {ep_inicio}/{epocas}.')
    print(f'Treinando {nome.upper()} | parâmetros: {sum(p.numel() for p in modelo.parameters()):,}')

    for ep in range(ep_inicio, epocas + 1):
        t0 = time.time()
        modelo.train()
        soma = 0.0
        for x, y in dl_treino:
            x, y = x.to(device), y.to(device)
            otim.zero_grad()
            loss = criterio(logits_de(modelo, x).reshape(-1, VOCAB_SIZE), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            otim.step()
            soma += loss.item()
        sched.step()
        val = avaliar(modelo, dl_val)
        registro = {
            'epoca': ep,
            'loss_treino': soma / len(dl_treino),
            'loss_val': val['loss'],
            'perplexidade_val': val['perplexidade'],
            'acuracia_val': val['acuracia'],
            'top5_val': val['top5'],
            'f1_macro_val': val['f1_macro'],
            'tempo_s': time.time() - t0,
        }
        historico.append(registro)
        print(f"Época {ep:02d}/{epocas} | treino {registro['loss_treino']:.4f} | val {val['loss']:.4f} | "
              f"ppl {val['perplexidade']:.2f} | acc {val['acuracia']:.4f} | top5 {val['top5']:.4f} | {registro['tempo_s']:.0f}s")
        if val['loss'] < melhor:
            melhor = val['loss']
            salvar_checkpoint(nome, modelo, caminho_melhor)
            print('  melhor modelo salvo no Drive')
        tmp = SSD / f'ultimo_{nome}.pt'
        torch.save({'epoca': ep, 'epocas_total': epocas, 'modelo': modelo.state_dict(),
                    'otimizador': otim.state_dict(), 'scheduler': sched.state_dict(),
                    'historico': historico, 'melhor_val': melhor, 'vocab_hash': VOCAB_HASH}, tmp)
        copiar_atomico(tmp, caminho_ultimo)
        salvar_historico_csv(nome, historico)
    return historico, True

def mascara_duracao():
    m = torch.zeros(VOCAB_SIZE, dtype=torch.bool)
    for t, i in token_para_id.items():
        if t.startswith('DURATION_'):
            m[i] = True
    return m.to(device)

def gerar_transformer(modelo, n_tokens=TOKENS_POR_MELODIA, temperatura=0.85, top_k=12, top_p=0.88, repeticao=1.35):
    modelo.eval()
    mask_dur = mascara_duracao()
    seq = [START_ID, token_para_id.get('NOTE_60', token_para_id.get('NOTE_64'))]
    tokens = [id_para_token[str(seq[1])]]
    ultimo = 'NOTE'
    with torch.no_grad():
        for _ in range(n_tokens):
            logits = logits_de(modelo, torch.tensor([seq[-MAX_SEQ_LEN:]], device=device))[0, -1].float().clone()
            for pid in set(seq[-8:]):
                logits[pid] = logits[pid] / repeticao if logits[pid] > 0 else logits[pid] * repeticao
            if ultimo == 'NOTE':
                logits[~mask_dur] = -float('inf')
                if ID_DUR_MIN is not None:
                    logits[ID_DUR_MIN] -= 2.0
            else:
                logits[mask_dur] = -float('inf')
            for especial in (PAD_ID, START_ID, END_ID):
                logits[especial] = -float('inf')
            logits = logits / max(temperatura, 0.01)
            if top_k > 0:
                limiar = torch.topk(logits, min(top_k, VOCAB_SIZE)).values[-1]
                logits[logits < limiar] = -float('inf')
            if 0.0 < top_p < 1.0:
                ordenados, idx = torch.sort(logits, descending=True)
                acumulada = torch.cumsum(F.softmax(ordenados, dim=-1), dim=-1)
                remover = acumulada > top_p
                remover[1:] = remover[:-1].clone()
                remover[0] = False
                logits[idx[remover]] = -float('inf')
            pid = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
            tok = id_para_token[str(pid)]
            tokens.append(tok)
            seq.append(pid)
            ultimo = 'NOTE' if tok.startswith('NOTE_') else ('DURATION' if tok.startswith('DURATION_') else 'REST')
    return tokens

def gerar_lstm(modelo, n_tokens=TOKENS_POR_MELODIA, temperatura=0.9, top_k=10):
    modelo.eval()
    tokens = []
    with torch.no_grad():
        logits, hidden = modelo(torch.tensor([[START_ID]], device=device))
        for _ in range(n_tokens):
            last = logits[:, -1, :] / max(temperatura, 0.01)
            last[:, PAD_ID] = -float('inf')
            last[:, START_ID] = -float('inf')
            vals, idxs = torch.topk(last, min(top_k, VOCAB_SIZE))
            escolhido = idxs.gather(-1, torch.multinomial(F.softmax(vals, dim=-1), 1)).item()
            tok = id_para_token[str(escolhido)]
            if tok == '<END>':
                break
            tokens.append(tok)
            logits, hidden = modelo(torch.tensor([[escolhido]], device=device), hidden)
    return tokens

def stats_melodia(tokens):
    notas = [int(t[5:]) for t in tokens if t.startswith('NOTE_')]
    if len(notas) < 2:
        return None
    cont = Counter(notas)
    intervalos = [abs(b - a) for a, b in zip(notas, notas[1:])]
    return {
        'n_notas': len(notas),
        'alturas_distintas': len(cont),
        'participacao_nota_dominante': max(cont.values()) / len(notas),
        'amplitude_semitons': max(notas) - min(notas),
        'intervalo_medio_semitons': float(np.mean(intervalos)),
        'pct_repeticoes_consecutivas': sum(1 for i in intervalos if i == 0) / len(intervalos),
    }

def media_stats(lista):
    validos = [s for s in lista if s]
    return {k: float(np.mean([s[k] for s in validos])) for k in validos[0]} if validos else {}

def tokens_reais_teste(n=500):
    saida = []
    for r in ler_jsonl_gz(pasta_cons / 'teste.jsonl.gz'):
        saida.append(r['tokens'])
        if len(saida) >= n:
            break
    return saida

def finalizar_modelo(nome, gerador):
    modelo = criar_modelo(nome)
    ckpt = torch.load(PASTA_MODELOS / f'modelo_{nome}.pt', map_location=device, weights_only=False)
    modelo.load_state_dict(ckpt['state_dict'])
    teste = avaliar(modelo, DataLoader(JanelasDataset(dados_teste), batch_size=BATCH_SIZE), calcular_auc=True)
    torch.manual_seed(42)
    melodias = [gerador(modelo) for _ in range(N_MELODIAS_GERADAS)]
    return {
        'teste': teste,
        'geracao': media_stats([stats_melodia(m) for m in melodias]),
        'real_teste': media_stats([stats_melodia(t) for t in tokens_reais_teste()]),
        'exemplo_tokens': melodias[0][:24],
    }

def salvar_resultado(nome, historico, resultado):
    dados = {'modelo': nome, 'vocabulario_tamanho': VOCAB_SIZE, 'historico': historico, **resultado}
    escrever_texto_atomico(PASTA_RELATORIOS / f'resultados_{nome}.json', json.dumps(dados, ensure_ascii=False, indent=2))
    t = resultado['teste']
    print(f"Teste {nome}: loss {t['loss']:.4f} | ppl {t['perplexidade']:.2f} | acc {t['acuracia']:.4f} | "
          f"top5 {t['top5']:.4f} | F1 macro {t['f1_macro']:.4f} | AUC {t['auc_roc_ovr']:.4f}")
    print('Estatísticas salvas em:', PASTA_RELATORIOS / f'resultados_{nome}.json')

def treinar_e_avaliar(nome, epocas, gerador):
    historico, treinou_agora = treinar_modelo(nome, epocas)
    if not treinou_agora and (PASTA_RELATORIOS / f'resultados_{nome}.json').exists():
        print(f'{nome}: resultados já calculados em relatorios/resultados_{nome}.json; avaliação não repetida.')
        return
    salvar_resultado(nome, historico, finalizar_modelo(nome, gerador))

print('Funções prontas.')

---
## 12. Treinar o Transformer (GPU T4)
Checkpoint em `modelos/modelo_transformer.pt` (melhor época) e `modelos/ultimo_transformer.pt` (retomada); histórico em `relatorios/historico_transformer.csv`; resultados em `relatorios/resultados_transformer.json`.

In [ ]:
treinar_e_avaliar('transformer', EPOCAS_TRANSFORMER, gerar_transformer)

---
## 13. Treinar a LSTM (GPU T4)
Checkpoint em `modelos/modelo_lstm.pt` (melhor época) e `modelos/ultimo_lstm.pt` (retomada); histórico em `relatorios/historico_lstm.csv`; resultados em `relatorios/resultados_lstm.json`.

In [ ]:
treinar_e_avaliar('lstm', EPOCAS_LSTM, gerar_lstm)

---
## 14. Estatísticas Comparativas
Gera o gráfico das curvas de treinamento e a tabela comparando os dois modelos, com as melodias reais do teste como referência.

In [ ]:
import pandas as pd

resultados = {}
for nome in ['transformer', 'lstm']:
    caminho = PASTA_RELATORIOS / f'resultados_{nome}.json'
    if caminho.exists():
        resultados[nome] = json.loads(caminho.read_text(encoding='utf-8'))

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for nome, r in resultados.items():
    ep = [h['epoca'] for h in r['historico']]
    eixos[0].plot(ep, [h['loss_treino'] for h in r['historico']], label=f'{nome} treino')
    eixos[0].plot(ep, [h['loss_val'] for h in r['historico']], '--', label=f'{nome} validação')
    eixos[1].plot(ep, [h['acuracia_val'] for h in r['historico']], label=nome)
eixos[0].set_title('Loss por época')
eixos[0].set_xlabel('época')
eixos[1].set_title('Acurácia (validação) por época')
eixos[1].set_xlabel('época')
for eixo in eixos:
    eixo.legend()
fig.tight_layout()
fig.savefig(SSD / 'curvas_treinamento.png', dpi=120)
copiar_atomico(SSD / 'curvas_treinamento.png', PASTA_RELATORIOS / 'curvas_treinamento.png')
plt.show()

linhas = []
for nome, r in resultados.items():
    t = r['teste']
    linhas.append({'modelo': nome, 'loss_teste': t['loss'], 'perplexidade': t['perplexidade'],
                   'acuracia': t['acuracia'], 'top5': t['top5'], 'precisao_macro': t['precisao_macro'],
                   'recall_macro': t['recall_macro'], 'f1_macro': t['f1_macro'], 'auc_roc_ovr': t['auc_roc_ovr']})
tabela_teste = pd.DataFrame(linhas).set_index('modelo')
print('Desempenho no conjunto de teste')
display(tabela_teste.round(4))

linhas_g = []
for nome, r in resultados.items():
    linhas_g.append({'fonte': nome, **r['geracao']})
linhas_g.append({'fonte': 'real (teste)', **next(iter(resultados.values()))['real_teste']})
tabela_geracao = pd.DataFrame(linhas_g).set_index('fonte')
print('Estatísticas das melodias (geradas x reais)')
display(tabela_geracao.round(3))

escrever_texto_atomico(PASTA_RELATORIOS / 'tabela_comparativa.csv',
                       tabela_teste.round(6).to_csv() + '\n' + tabela_geracao.round(6).to_csv())
print('Gráfico e tabelas salvos em:', PASTA_RELATORIOS)

---
## 15. Conferência dos Arquivos Persistidos no Drive

In [ ]:
for pasta in [PASTA_DADOS, PASTA_MODELOS, PASTA_RELATORIOS]:
    arquivos = [a for a in pasta.rglob('*') if a.is_file() and '_arquivo' not in a.parts]
    print(f'{pasta}: {len(arquivos)} arquivos')
print('Checkpoints:', [p.name for p in PASTA_MODELOS.glob('*.pt')])
print('Dados no formato do repositório:', ZIP_DADOS if ZIP_DADOS.exists() else '(não gerado)')

---
## 16. Baixar os resultados
Empacota os **modelos** (`modelo_*.pt`) e **todos os relatórios** em um `.zip` e inicia o download no navegador. O mesmo `.zip` fica salvo em `downloads/` no Drive. A estrutura interna (`modelos/` e `relatorios/`) pode ser extraída direto na raiz do projeto.

Se o navegador bloquear o download, permita downloads para o `colab.research.google.com` e rode só esta célula de novo.

In [ ]:
from google.colab import files

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_resultados = SSD / f'pi6_resultados_{ts}.zip'
with zipfile.ZipFile(zip_resultados, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(PASTA_MODELOS.glob('modelo_*.pt')):
        zf.write(p, f'modelos/{p.name}')
    for p in sorted(PASTA_RELATORIOS.iterdir()):
        if p.is_file():
            zf.write(p, f'relatorios/{p.name}')
copiar_atomico(zip_resultados, DRIVE / 'downloads' / zip_resultados.name)
print(f'Pacote: {zip_resultados.name} ({zip_resultados.stat().st_size / 1e6:.1f} MB)')
with zipfile.ZipFile(zip_resultados) as zf:
    for info in zf.infolist():
        print(f'  {info.filename}  ({info.file_size / 1e6:.2f} MB)')
files.download(str(zip_resultados))